# Neo4j + Spark
Run `sh scripts/platform.sh load-neo4j` first (needs the `neo4j` module in `COMPOSE_PROFILES`).

If the kernel seems to hang after running the cell below, that's the Neo4j driver's own
background threads -- the query itself has already finished. `Kernel > Restart Kernel` clears it.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

pw = os.environ["NEO4J_PASSWORD"]
spark = SparkSession.builder.appName("neo4j-notebook").enableHiveSupport().getOrCreate()

def neo(**extra):
    r = (spark.read.format("org.neo4j.spark.DataSource")
         .option("url", "bolt://neo4j:7687")
         .option("authentication.basic.username", "neo4j")
         .option("authentication.basic.password", pw))
    for k, v in extra.items():
        r = r.option(k, v)
    return r.load()

print("NEO4J_CUSTOMERS", neo(labels=":Customer").count())

lines = neo(query="MATCH (o:Order {status: 'delivered'})-[c:CONTAINS]->(p:Product)-[:IN_CATEGORY]->(cat:Category) "
                  "RETURN cat.name AS category, c.quantity AS quantity, p.price_cents AS price_cents")
rows = (lines.groupBy("category")
        .agg(F.sum(F.col("quantity") * F.col("price_cents")).alias("cents"))
        .orderBy("category").collect())
fmt = lambda c: f"{int(c) // 100}.{int(c) % 100:02d}"
for r in rows:
    print("CATEGORY", r["category"], fmt(r["cents"]))
print("TOTAL", fmt(sum(int(r["cents"]) for r in rows)))
spark.stop()